In [ ]:
from roboflow import Roboflow
# 1808 dataset
print("Downloading dataset 1...")

# !! Consider using environment variables or a config file for API keys !!
rf_1 = Roboflow(api_key="5XEfDZRIIjMqbazGL6nS")
project_1 = rf_1.workspace("bachelor-d6dut").project("car-detection-bachelor")
version_1 = project_1.version(3)
dataset_1 = version_1.download("yolov8")
download_location_1 = dataset_1.location
print(f"Dataset 1 downloaded to: {download_location_1}")


In [ ]:
# 998 dataset
print("Downloading dataset 2...")
# !! Consider using environment variables or a config file for API keys !!
rf_2 = Roboflow(api_key="zolhGuu1snsuy5zeAJsS")
project_2 = rf_2.workspace("bachelor-jvyhh").project("superstar-car-project")
version_2 = project_2.version(2)
dataset_2 = version_2.download("yolov8")
download_location_2 = dataset_2.location
print(f"Dataset 2 downloaded to: {download_location_2}")


In [ ]:
# 248 dataset
print("Downloading dataset 3...")
# !! Consider using environment variables or a config file for API keys !!
rf_3 = Roboflow(api_key="VcIz6aIo01dcvjkU8AMO")
project_3 = rf_3.workspace("labeling-2").project("labeling-2-hoswc")
version_3 = project_3.version(1)
dataset_3 = version_3.download("yolov8")
download_location_3 = dataset_3.location
print(f"Dataset 3 downloaded to: {download_location_3}")


In [ ]:
# 707 dataset
print("Downloading dataset 4...")
# !! Consider using environment variables or a config file for API keys !!
rf_4 = Roboflow(api_key="uxiGJrqUcSW3MVh0ONYd")
project_4 = rf_4.workspace("poopy-epswg").project("my-first-project-wajtf")
version_4 = project_4.version(3)
dataset_4 = version_4.download("yolov8")
download_location_4 = dataset_4.location
print(f"Dataset 4 downloaded to: {download_location_4}")

# Consolidate paths
downloaded_dataset_paths = [download_location_1, download_location_2, download_location_3, download_location_4]


In [ ]:
import os
import shutil
import random
import yaml

# Output merged dataset directory name
merged_dir_name = "merged_dataset"
# Get the absolute path for the merged directory based on the current working directory
# Note: This path will depend on where you run the notebook!
merged_dir = os.path.abspath(merged_dir_name)

# Temporary directories for collecting files before splitting
merged_images_temp = os.path.join(merged_dir, "images_temp")
merged_labels_temp = os.path.join(merged_dir, "labels_temp")

# Create temporary merged folders if they don't exist
print(f"Creating temporary directories in: {merged_dir}")
os.makedirs(merged_images_temp, exist_ok=True)
os.makedirs(merged_labels_temp, exist_ok=True)

# Function to copy files from each downloaded dataset's splits
def copy_dataset_files(source_dataset_folder, temp_img_folder, temp_lbl_folder):
    print(f"  Copying from: {source_dataset_folder}")
    copied_count = 0
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(source_dataset_folder, split, 'images')
        lbl_dir = os.path.join(source_dataset_folder, split, 'labels')

        if os.path.exists(img_dir):
            for file in os.listdir(img_dir):
                try:
                    # Add a check to prevent copying over existing files with the same name
                    # This could happen if datasets have identically named images
                    destination_file = os.path.join(temp_img_folder, file)
                    if not os.path.exists(destination_file):
                         shutil.copy(os.path.join(img_dir, file), destination_file)
                         copied_count += 1
                    # else:
                    #    print(f"    Skipping duplicate image file: {file}") # Optional: uncomment to see duplicates
                except shutil.SameFileError:
                    # Handle cases where the source and destination might be the same (less likely here)
                    pass
                except Exception as e:
                    print(f"    Error copying image {file}: {e}")

        if os.path.exists(lbl_dir):
            for file in os.listdir(lbl_dir):
                try:
                    destination_file = os.path.join(temp_lbl_folder, file)
                    if not os.path.exists(destination_file):
                        shutil.copy(os.path.join(lbl_dir, file), destination_file)
                    # else:
                    #    print(f"    Skipping duplicate label file: {file}") # Optional: uncomment to see duplicates
                except shutil.SameFileError:
                    pass
                except Exception as e:
                    print(f"    Error copying label {file}: {e}")
    print(f"  Copied {copied_count} new image files (labels copied accordingly).")

# Copy files from all downloaded datasets into the temporary folders
print("Starting file copy process...")
for dataset_path in downloaded_dataset_paths:
    # Roboflow download gives the path to the specific version folder
    copy_dataset_files(dataset_path, merged_images_temp, merged_labels_temp)
print("Finished copying files to temporary location.")


In [ ]:
# Get list of all image files collected in the temporary folder
image_files = [f for f in os.listdir(merged_images_temp) if os.path.isfile(os.path.join(merged_images_temp, f))]
print(f"Total unique image files found in temp folder: {len(image_files)}")

# Shuffle the image files randomly
random.shuffle(image_files)

# Define split ratios
train_ratio = 0.70
val_ratio = 0.15
# test_ratio is implicitly 1.0 - train_ratio - val_ratio

# Calculate split indices
total_files = len(image_files)
train_split_idx = int(train_ratio * total_files)
val_split_idx = train_split_idx + int(val_ratio * total_files)

# Create file lists for each split
train_files = image_files[:train_split_idx]
val_files = image_files[train_split_idx:val_split_idx]
test_files = image_files[val_split_idx:]

print(f"Splitting into: {len(train_files)} train, {len(val_files)} validation, {len(test_files)} test images.")

# Create final split directories (train/images, train/labels, etc.) inside the main merged_dir
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(merged_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(merged_dir, split, 'labels'), exist_ok=True)

# Function to move files from temporary folders to final split folders
def move_files_to_split(file_list, split_name):
    print(f"  Moving files to '{split_name}' split...")
    moved_count = 0
    missing_labels = 0
    for image_file in file_list:
        base_name, _ = os.path.splitext(image_file)
        label_file = base_name + '.txt'

        img_src = os.path.join(merged_images_temp, image_file)
        lbl_src = os.path.join(merged_labels_temp, label_file)

        # Use the absolute path stored in merged_dir
        img_dst = os.path.join(merged_dir, split_name, "images", image_file)
        lbl_dst = os.path.join(merged_dir, split_name, "labels", label_file)

        # Move image
        if os.path.exists(img_src):
            try:
                shutil.move(img_src, img_dst)
                moved_count += 1
            except Exception as e:
                print(f"    Error moving image {image_file} to {img_dst}: {e}")
        else:
            # This shouldn't happen if the image_files list is correct
            print(f"    Warning: Image source file not found during move: {img_src}")

        # Move corresponding label if it exists
        if os.path.exists(lbl_src):
            try:
                shutil.move(lbl_src, lbl_dst)
            except Exception as e:
                print(f"    Error moving label {label_file} to {lbl_dst}: {e}")
        else:
            # It's okay if some images don't have labels (e.g., background/negative images)
            missing_labels += 1
            # print(f"    Note: Label source file not found: {lbl_src}") # Optional: uncomment for debugging

    print(f"  Moved {moved_count} images to '{split_name}'. {missing_labels} images had no corresponding label file.")

# Move the files according to the splits
print("Moving files into final split directories...")
move_files_to_split(train_files, "train")
move_files_to_split(val_files, "valid")
move_files_to_split(test_files, "test")

# Clean up the temporary merged images/labels folders
print("Cleaning up temporary folders...")
try:
    shutil.rmtree(merged_images_temp)
    shutil.rmtree(merged_labels_temp)
    print(f"  Temporary folders removed: {merged_images_temp} and {merged_labels_temp}")
except Exception as e:
    print(f"  Error removing temporary folders: {e}")


In [ ]:
# --- IMPORTANT: You MUST replace these with your actual class names in the correct order ---
# The order must match the class index used in your .txt label files (e.g., 0 for the first name, 1 for the second, etc.)
class_names = ['car'] # Example: ['person', 'car', 'traffic light']
# --- IMPORTANT: Double-check this list carefully! ---

# Define the structure for the data.yaml file
# Using relative paths from the YAML file location is standard for YOLO
data_yaml_content = {
    'path': merged_dir, # Absolute path to dataset root (useful info, YOLO might not use it directly)
    'train': 'train/images', # Relative path from 'path' or YAML file location
    'val': 'valid/images',   # Relative path from 'path' or YAML file location
    'test': 'test/images',  # Relative path from 'path' or YAML file location (optional for training)
    'nc': len(class_names),
    'names': class_names
}

# Define the absolute path for the data.yaml file within the merged dataset directory
# Use the absolute path stored in merged_dir
yaml_file_path = os.path.join(merged_dir, 'data.yaml')

# Write the dictionary to the YAML file
print(f"Creating data.yaml at: {yaml_file_path}")
try:
    with open(yaml_file_path, 'w') as f:
        # Use sort_keys=False to maintain the order specified in the dictionary
        yaml.dump(data_yaml_content, f, default_flow_style=None, sort_keys=False)
    print(f"  Successfully created data.yaml")
except Exception as e:
    print(f"  Error creating data.yaml: {e}")

print("\n---")
print(f"✅ Dataset merged and split into: {merged_dir}")
print(f"   Train: {len(train_files)} images")
print(f"   Valid: {len(val_files)} images")
print(f"   Test:  {len(test_files)} images")
print(f"✅ data.yaml created at: {yaml_file_path}")
print("👉 IMPORTANT: Please verify the 'names' list in data.yaml matches your dataset classes and their order!")
